# Target Base Reconciliation
**Merchant 501 · October 2026 · Diwali Campaigns**

---

This notebook derives the Finance `target_base` metric for Merchant 501's October 2026 campaign sends.

The methodology proceeds in two stages:
1. **Top-down reconciliation** — start from all raw rows and progressively apply reporting filters (Steps 0–2) to arrive at the final number.
2. **Bottom-up verification** — independently compute the count per underlying communication using the business definition (retry-chain vs. standalone logic), and confirm the totals match.

## Setup

In [ ]:
import sqlite3
import pandas as pd
from IPython.display import display, Markdown

DB_PATH = "../data/comm_log.db"
conn = sqlite3.connect(DB_PATH)

def run_sql(query, conn=conn):
    """Execute a SQL query and return results as a styled DataFrame."""
    return pd.read_sql_query(query, conn)

print(f"Connected to: {DB_PATH}")

## 1 · Data Exploration

Before computing anything, let us inspect both tables to understand the data landscape.

In [ ]:
display(Markdown("### Campaign Table"))
campaigns = run_sql("SELECT * FROM campaign ORDER BY id")
display(campaigns)

In [ ]:
display(Markdown("### Communication Log Table"))
comm_log = run_sql("SELECT * FROM communication_log ORDER BY id")
display(comm_log)

### Key observations from raw data

| Fact | Value |
|---|---|
| Total campaigns | 7 |
| Total communication-log rows | 30 |
| Merchant | 501 |
| Date range | October 2026 |
| Channel | SMS only |
| Communication type | `'2'` (Campaign) only |

---

## 2 · Top-Down Reconciliation (Steps 0–2)

We start with every row in `communication_log` and progressively apply three reporting filters. Each step is shown as an independent query, and then all three are combined into **a single CTE** that produces the final answer.

### Step 0 — Count all campaign communication-log rows

Starting point: every row is a send attempt for Merchant 501, October 2026, communication type `'2'`.

In [ ]:
step0 = run_sql("""
    SELECT COUNT(*) AS total_rows
    FROM communication_log
    WHERE merchant_id = 501
      AND communication_type = '2'
      AND sent_time >= '2026-10-01'
      AND sent_time <  '2026-11-01'
""")
display(step0)
print(f"→ Step 0 result: {step0.iloc[0, 0]}")

### Step 1 — Keep only successful deliveries

`delivery_status = 900` means delivered; `1100` means failed. We remove the failed rows.

In [ ]:
step1 = run_sql("""
    SELECT COUNT(*) AS delivered_rows
    FROM communication_log
    WHERE merchant_id = 501
      AND communication_type = '2'
      AND sent_time >= '2026-10-01'
      AND sent_time <  '2026-11-01'
      AND delivery_status = 900
""")
display(step1)
print(f"→ Step 1 result: {step1.iloc[0, 0]}  (30 − 4 failed = 26)")

In [ ]:
# Which rows failed?
display(Markdown("**Failed send attempts:**"))
failed = run_sql("""
    SELECT id, communication_id, customer_id, delivery_status
    FROM communication_log
    WHERE merchant_id = 501
      AND communication_type = '2'
      AND sent_time >= '2026-10-01'
      AND sent_time <  '2026-11-01'
      AND delivery_status = 1100
""")
display(failed)

### Step 2 — Exclude sends from non-reportable campaigns

A campaign is included in official reporting only when:
- `creation_status` is in the finalized set (`approved`, `aborted`, `resumed`, `stopped`)
- `processing_status = 'processed'`

Campaign **9004** is still `approval_awaiting` — its 4 sends are excluded.

In [ ]:
step2 = run_sql("""
    SELECT COUNT(*) AS reportable_delivered_rows
    FROM communication_log AS l
    JOIN campaign AS c
        ON c.id = l.communication_id
    WHERE l.merchant_id = 501
      AND l.communication_type = '2'
      AND l.sent_time >= '2026-10-01'
      AND l.sent_time <  '2026-11-01'
      AND l.delivery_status = 900
      AND c.creation_status IN ('approved', 'aborted', 'resumed', 'stopped')
      AND c.processing_status = 'processed'
""")
display(step2)
print(f"→ Step 2 result: {step2.iloc[0, 0]}  (26 − 4 from campaign 9004 = 22)")

In [ ]:
# Show what was excluded — Campaign 9004 rows
display(Markdown("**Excluded sends (campaign 9004 — `approval_awaiting`):**"))
excluded = run_sql("""
    SELECT l.id, l.communication_id, l.customer_id, l.delivery_status,
           c.creation_status, c.processing_status
    FROM communication_log AS l
    JOIN campaign AS c ON c.id = l.communication_id
    WHERE l.communication_id = 9004
""")
display(excluded)

### Combined CTE — Steps 0–2 as a Single Query (Final Answer)

The three steps above combine naturally into one query. This **is** the final answer from the top-down reconciliation.

In [ ]:
final_topdown = run_sql("""
    WITH

    -- Step 0: All campaign sends for merchant 501, Oct 2026
    step0_all_rows AS (
        SELECT *
        FROM communication_log
        WHERE merchant_id = 501
          AND communication_type = '2'
          AND sent_time >= '2026-10-01'
          AND sent_time <  '2026-11-01'
    ),

    -- Step 1: Keep only successful deliveries
    step1_delivered AS (
        SELECT *
        FROM step0_all_rows
        WHERE delivery_status = 900
    ),

    -- Step 2: Keep only sends from reportable campaigns
    step2_reportable AS (
        SELECT l.*
        FROM step1_delivered AS l
        JOIN campaign AS c
            ON c.id = l.communication_id
        WHERE c.creation_status IN ('approved', 'aborted', 'resumed', 'stopped')
          AND c.processing_status = 'processed'
    )

    SELECT
        (SELECT COUNT(*) FROM step0_all_rows)    AS step0_all_rows,
        (SELECT COUNT(*) FROM step1_delivered)    AS step1_delivered,
        (SELECT COUNT(*) FROM step2_reportable)   AS step2_reportable_target_base
""")

display(Markdown("### Reconciliation Summary"))
display(final_topdown)
print(f"\n✅ Final target_base (top-down) = {final_topdown.iloc[0, 2]}")

---

## 3 · Bottom-Up Verification

We now verify the result from the ground up using the **business definition** of `target_base`:

> *For a given underlying communication (a campaign plus every retry chained off it), how many distinct customers were reached? A standalone campaign's sends are each independent events.*

### Approach
1. Use a **recursive CTE** to find the root (parent) of every campaign
2. Classify each root as **standalone** or **retry-chain parent**
3. Count instances separately for each parent communication
4. Sum up all counts to verify the total

### 3.1 · Identify campaign families via recursive CTE

In [ ]:
families = run_sql("""
    WITH RECURSIVE campaign_roots AS (
        -- Base: campaigns with no parent are their own root
        SELECT id AS campaign_id, id AS root_id
        FROM campaign
        WHERE parent_id IS NULL

        UNION ALL

        -- Recursive: follow the retry chain upward
        SELECT c.id AS campaign_id, cr.root_id
        FROM campaign AS c
        JOIN campaign_roots AS cr
            ON c.parent_id = cr.campaign_id
    )
    SELECT
        cr.campaign_id,
        cr.root_id,
        c.name,
        c.parent_id,
        c.creation_status,
        c.processing_status
    FROM campaign_roots AS cr
    JOIN campaign AS c ON c.id = cr.campaign_id
    ORDER BY cr.root_id, cr.campaign_id
""")

display(Markdown("### Campaign Family Tree"))
display(families)

**Campaign structure:**

```
9001 (Diwali Cart Recovery - Wave 1)
├── 9002 (Retry A)
│   └── 9003 (Retry B)
└── 9004 (Retry C — ⚠ approval_awaiting, EXCLUDED)

9101 (Diwali Flash Sale — Standalone, no retries)

9201 (Diwali Wave 2)
└── 9202 (Retry)
```

### 3.2 · Classify standalone vs. retry-chain

In [ ]:
classification = run_sql("""
    SELECT
        c.id,
        c.name,
        c.parent_id,
        CASE
            WHEN c.parent_id IS NULL
             AND NOT EXISTS (
                 SELECT 1 FROM campaign AS child
                 WHERE child.parent_id = c.id
             )
            THEN 'STANDALONE'
            ELSE 'RETRY CHAIN'
        END AS classification
    FROM campaign AS c
    ORDER BY c.id
""")

display(Markdown("### Campaign Classification"))
display(classification)

### 3.3 · Per-parent communication counts

For each parent communication, we count qualifying sends using the appropriate rule and print the result.

In [ ]:
per_parent = run_sql("""
    WITH RECURSIVE campaign_roots AS (
        SELECT id AS campaign_id, id AS root_id
        FROM campaign
        WHERE parent_id IS NULL
        UNION ALL
        SELECT c.id AS campaign_id, cr.root_id
        FROM campaign AS c
        JOIN campaign_roots AS cr ON c.parent_id = cr.campaign_id
    ),
    eligible_sends AS (
        SELECT l.communication_id, l.customer_id, cr.root_id,
            CASE WHEN c.parent_id IS NULL
                 AND NOT EXISTS (SELECT 1 FROM campaign AS child WHERE child.parent_id = c.id)
                THEN 1 ELSE 0
            END AS is_standalone
        FROM communication_log AS l
        JOIN campaign AS c ON c.id = l.communication_id
        JOIN campaign_roots AS cr ON cr.campaign_id = c.id
        WHERE l.merchant_id = 501
          AND l.communication_type = '2'
          AND l.sent_time >= '2026-10-01'
          AND l.sent_time <  '2026-11-01'
          AND l.delivery_status = 900
          AND c.creation_status IN ('approved','aborted','resumed','stopped')
          AND c.processing_status = 'processed'
    ),
    family_counts AS (
        SELECT root_id,
            CASE WHEN MAX(is_standalone) = 1
                THEN 'Standalone (COUNT(*))'
                ELSE 'Retry Chain (COUNT DISTINCT)'
            END AS counting_rule,
            CASE WHEN MAX(is_standalone) = 1
                THEN COUNT(*)
                ELSE COUNT(DISTINCT customer_id)
            END AS qualifying_sends
        FROM eligible_sends
        GROUP BY root_id
    )
    SELECT root_id, counting_rule, qualifying_sends
    FROM family_counts
    ORDER BY root_id
""")

display(Markdown("### Qualifying sends per parent communication"))
display(per_parent)

total = per_parent['qualifying_sends'].sum()
parts = ' + '.join(str(v) for v in per_parent['qualifying_sends'])
print(f"\n→ Total: {parts} = {total}")

---

## 4 · Final Answer

Both methods converge on the same number:

In [ ]:
final = run_sql("""
    WITH RECURSIVE campaign_roots AS (
        SELECT id AS campaign_id, id AS root_id
        FROM campaign
        WHERE parent_id IS NULL
        UNION ALL
        SELECT c.id AS campaign_id, cr.root_id
        FROM campaign AS c
        JOIN campaign_roots AS cr
            ON c.parent_id = cr.campaign_id
    ),
    eligible_sends AS (
        SELECT
            l.communication_id, l.customer_id, cr.root_id,
            CASE
                WHEN c.parent_id IS NULL
                 AND NOT EXISTS (
                     SELECT 1 FROM campaign AS child
                     WHERE child.parent_id = c.id
                 )
                THEN 1 ELSE 0
            END AS is_standalone
        FROM communication_log AS l
        JOIN campaign AS c ON c.id = l.communication_id
        JOIN campaign_roots AS cr ON cr.campaign_id = c.id
        WHERE l.merchant_id = 501
          AND l.communication_type = '2'
          AND l.sent_time >= '2026-10-01'
          AND l.sent_time <  '2026-11-01'
          AND l.delivery_status = 900
          AND c.creation_status IN ('approved', 'aborted', 'resumed', 'stopped')
          AND c.processing_status = 'processed'
    ),
    family_counts AS (
        SELECT root_id,
            CASE WHEN MAX(is_standalone) = 1
                THEN COUNT(*)
                ELSE COUNT(DISTINCT customer_id)
            END AS qualifying_sends
        FROM eligible_sends
        GROUP BY root_id
    )
    SELECT SUM(qualifying_sends) AS target_base
    FROM family_counts
""")

display(Markdown(f"# `target_base = {final.iloc[0, 0]}`"))

### Reconciliation Bridge

| Step | Description | Result | Reason |
|-----:|:------------|-------:|:-------|
| **0** | Naive count of campaign communication-log rows | **30** | Initial gross baseline: queried all 30 campaign send attempts recorded in `communication_log` for merchant 501 during October 2026. |
| **1** | Keep delivered sends only | **26** | Filtered for `delivery_status = 900`. Excluded 4 soft-failed attempts (status 1100: C2, C3×2, D1) which consumed credits but never reached customers. |
| **2** | Exclude sends from non-reportable campaigns | **22** | Enforced campaign lifecycle governance (`creation_status` finalized AND `processing_status = 'processed'`). Excluded 4 delivered sends from Campaign 9004 because it executed ahead of approval (still `approval_awaiting`). |
| **3** | Validate retry-family counting | **22** | Investigated retry-chain deduplication: audited delivery logs for families 9001 and 9201. Because previous attempts failed and only retries succeeded, each customer (C1–C10, D1–D5) had exactly one delivered send; zero redundant deliveries exist to deduct, confirming 10 + 5 = 15 unique reached customers. |
| **4** | Validate standalone-campaign counting | **22** | Investigated customer duplicates: customer C20 was delivered twice under Campaign 9101 (Oct 10 & Oct 20). Hierarchy check confirmed 9101 is standalone without retries; domain rules treat repeated standalone sends as independent audience re-targets, confirming all 7 sends qualify without deduplication. |
| **Final** | **Finance `target_base`** | **22** | Independent ground-up validation via recursive CTE confirms: 10 distinct customers (9001 chain) + 7 standalone send events (9101) + 5 distinct customers (9201 chain) = **22** qualifying sends. |


In [ ]:
conn.close()
print("Database connection closed.")